# 01. Baseline adapter evaluated on the gold set (Phase 2)
Runtime → Change runtime type → **T4 GPU**. Run cells **1-7 in order**.

Uses the project repo and branch (`feat/codebook-data-curation`), downloads baseline adapter, runs it on the 100 gold titles and scores it.
Final baseline result: **F1 0.44, exact title 30/100** (see the project report). Saved in `results/baseline_v0/`. This stage is done; the notebook is kept as a trace of the work.

In [ ]:
# Clones the project repo (or pulls the latest commit) and installs dependencies. Run it first in every new session.
# PHASE: shared by all stages.
# 1. Clone (or update) the repo and install dependencies
REPO_URL = "https://github.com/m-mirkan/peft-cpe-extraction-pipeline.git"
BRANCH = "feat/codebook-data-curation"

import os
if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt gdown

import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Downloads baseline LoRA adapter from Google Drive into checkpoints/qwen25_3b_cpe_lora.
# PHASE 2 (baseline).
# 2. Download baseline adapter
import os, zipfile
LORA_GDRIVE_ID = "1eKyqTdDT36zPEpi6y62kZg9V8vTmJVaX"
LORA_DIR = "checkpoints/qwen25_3b_cpe_lora"
os.makedirs(LORA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(LORA_DIR, "adapter_config.json")):
    !gdown {LORA_GDRIVE_ID} -O lora_adapter.zip
    with zipfile.ZipFile("lora_adapter.zip", "r") as z:
        z.extractall(LORA_DIR)
    entries = os.listdir(LORA_DIR)
    if len(entries) == 1 and os.path.isdir(os.path.join(LORA_DIR, entries[0])):
        nested = os.path.join(LORA_DIR, entries[0])
        for f in os.listdir(nested):
            os.rename(os.path.join(nested, f), os.path.join(LORA_DIR, f))
        os.rmdir(nested)
print("LoRA adapter files:", os.listdir(LORA_DIR))

In [ ]:
# Checks that the gold files and the patched inference script are in the repo.
# PHASE 2 (baseline). Safe to run again, takes a second.
# 3. Sanity check: the gold files and the patched script must be there
import json
!ls data/gold scripts
inputs = json.load(open("data/gold/gold_inputs.json", encoding="utf-8"))
gold = json.load(open("data/gold/gold_template.json", encoding="utf-8"))
print("gold titles:", len(inputs), "| unique:", len(set(inputs)), "| labeled:", sum(g["status"] == "labeled" for g in gold))
assert len(inputs) == 100 and [g["input"] for g in gold] == inputs
print("patched inference script (should print 2 lines):")
!grep -n "max_new_tokens=2048\|FAILED" src/infer_and_resolve.py

In [ ]:
# Runs baseline adapter on gold titles 1-50 and saves predictions after every title.
# PHASE 2 (baseline), already done. Do NOT rerun for the new stage.
# 4. Run titles 1-50 (saves after every title; failures are recorded, not fatal)
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/gold/gold_inputs.json --offset 0 --limit 50 --output_file outputs/gold_base_p1.json

In [ ]:
# Runs baseline adapter on gold titles 51-100.
# PHASE 2 (baseline), already done. Do NOT rerun for the new stage.
# 5. Run titles 51-100
!python src/infer_and_resolve.py --lora_dir checkpoints/qwen25_3b_cpe_lora --input_file data/gold/gold_inputs.json --offset 50 --limit 50 --output_file outputs/gold_base_p2.json

In [ ]:
# Scores the baseline predictions against the gold set (precision, recall, F1, part accuracy, exact title).
# PHASE 2 (baseline), already done. Final baseline result: F1 0.44 (see the report).
# 6. Score against the gold set
import json
n1 = len(json.load(open("outputs/gold_base_p1.json", encoding="utf-8")))
n2 = len(json.load(open("outputs/gold_base_p2.json", encoding="utf-8")))
print("predictions saved:", n1, "+", n2, "(expected 50 + 50)")
!python scripts/score_gold.py --gold data/gold/gold_template.json --pred baseline=outputs/gold_base_p1.json,outputs/gold_base_p2.json --json_out outputs/gold_base_scores.json

In [ ]:
# Downloads the baseline prediction and score files to your computer.
# PHASE 2 (baseline), already done.
# 7. Download the outputs
from google.colab import files
files.download("outputs/gold_base_p1.json")
files.download("outputs/gold_base_p2.json")
files.download("outputs/gold_base_scores.json")